In [57]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [58]:
import numpy as np
import pandas as pd

from DQR import DQR_continuous, DQR_categorical

pd.set_option('display.max_colwidth', None)

# Read in data

In [59]:
df = pd.read_csv("../dataset/networkTraffic.csv", index_col=0, na_values="?")

In [60]:
df

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat
id,,,,,,,,,,,,,,,,,,,,,
1,0.000011,udp,NaN,INT,2,0,496,0,90909.090200,254,...,1,1,2,0,0,0,1,2,0,0
2,0.000008,udp,NaN,INT,2,0,1762,0,125000.000300,254,...,1,1,2,0,0,0,1,2,0,0
3,0.000005,udp,NaN,INT,2,0,1068,0,200000.005100,254,...,1,1,3,0,0,0,1,3,0,0
4,0.000006,udp,NaN,INT,2,0,900,0,166666.660800,254,...,2,1,3,0,0,0,2,3,0,0
5,0.000010,udp,NaN,INT,2,0,2126,0,100000.002500,254,...,2,1,3,0,0,0,2,3,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257669,0.000009,udp,dns,INT,2,0,114,0,111111.107200,254,...,24,13,24,0,0,0,24,24,0,9
257670,0.505762,tcp,NaN,FIN,10,8,620,354,33.612649,254,...,1,1,2,0,0,0,1,1,0,8
257671,0.000009,udp,dns,INT,2,0,114,0,111111.107200,254,...,3,3,13,0,0,0,3,12,0,9


In [61]:
continuous_feature_names = ['dur', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'swin', 'dwin', 'sloss', 'dloss', 'sload', 'spkts', 'dpkts', 'dload', 'stcpb',
                            'dtcpb', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'sjit', 'djit', 'sinpkt', 'dinpkt', 'tcprtt',
                            'synack', 'ackdat', 'ct_flw_http_mthd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm',
                            'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm']
categorical_feature_names = [feature for feature in df.columns if feature not in continuous_feature_names and feature != 'attack_cat']
target_feature = df['attack_cat']

# DQR

In [62]:
all_continuous_features = df.loc[:, continuous_feature_names]
all_categorical_features = df.loc[:, categorical_feature_names]

In [63]:
DQR_cont = DQR_continuous(all_continuous_features)
DQR_cat = DQR_categorical(all_categorical_features)

In [64]:
DQR_cont

,Count,% Miss.,Card.,Min.,1st Qrt.,Mean,Median,3rd Qrt.,Max.,Std. Dev.
dur,257673,0.0,109945,0.0,0.000008,1.246715e+00,0.004285,6.857770e-01,5.999999e+01,5.974305e+00
sbytes,257673,0.0,9382,24.0,114.000000,8.572952e+03,528.000000,1.362000e+03,1.435577e+07,1.737739e+05
dbytes,257673,0.0,8653,0.0,0.000000,1.438729e+04,178.000000,1.064000e+03,1.465753e+07,1.461993e+05
rate,257673,0.0,115763,0.0,30.789277,9.125391e+04,2955.664893,1.250000e+05,1.000000e+06,1.603446e+05
sttl,257673,0.0,13,0.0,62.000000,1.800009e+02,254.000000,2.540000e+02,2.550000e+02,1.024883e+02
dttl,257673,0.0,9,0.0,0.000000,8.475496e+01,29.000000,2.520000e+02,2.540000e+02,1.127621e+02
swin,257673,0.0,22,0.0,0.000000,1.217537e+02,0.000000,2.550000e+02,2.550000e+02,1.273674e+02
dwin,257673,0.0,19,0.0,0.000000,1.192546e+02,0.000000,2.550000e+02,2.550000e+02,1.272305e+02
sloss,257673,0.0,490,0.0,0.000000,4.889317e+00,0.000000,3.000000e+00,5.319000e+03,6.557495e+01
dloss,257673,0.0,476,0.0,0.000000,6.743691e+00,0.000000,2.000000e+00,5.507000e+03,5.370222e+01


In [65]:
DQR_cat

,Count,% Miss.,Card.,Mode,Mode Freq.,Mode %,2nd Mode,2nd Mode Freq.,2nd Mode %
proto,257673,0.000000,133,tcp,123041,47.750831,udp,92701,35.976218
service,257673,54.845094,12,dns,68661,59.011448,http,27011,23.214900
state,257673,0.000000,11,FIN,117164,45.470034,INT,116438,45.188281
ct_state_ttl,257673,0.000000,7,2,116154,45.078064,1,76589,29.723332
is_ftp_login,257673,0.000000,4,0,254428,98.740652,1,3219,1.249258
ct_ftp_cmd,257673,0.000000,4,0,254426,98.739876,1,3215,1.247705
is_sm_ips_ports,257673,0.000000,2,0,253995,98.572609,1,3678,1.427391


# Removing correlated features

In [66]:
continuous_feature_names = [feature for feature in continuous_feature_names if feature not in ['dwin', 'sbytes', 'dbytes', 'spkts', 'dpkts', 'dwin', 'tcprtt', 'ct_srv_dst', 'ct_src_dport_ltm', 'dmean']]
continuous_features = df.loc[:, continuous_feature_names]

In [67]:
categorical_feature_names = [feature for feature in categorical_feature_names if feature not in ['ct_ftp_cmd']]
categorical_features = df.loc[:, categorical_feature_names]

# Write datasets

In [71]:
continuous_features.to_csv("../dataset/pre-processed data/continuous_features.csv", index=False)
categorical_features.to_csv("../dataset/pre-processed data/categorical_features.csv", index=False)
target_feature.to_csv("../dataset/pre-processed data/target_feature.csv", index=False)